# Lesson 10: Difference-in-Differences

## Opening Story: California Tobacco Control

In 1988, California passed Proposition 99, which increased the cigarette tax by 25 cents per pack and funded anti-smoking campaigns. How much did this reduce smoking?

The difference-in-differences (DiD) method compares changes in California's smoking rates before and after the policy to changes in other states that didn't implement similar policies. This accounts for time trends and other factors that affected all states.

---

## Learning Objectives

By the end of this lesson, you should be able to:

1. Explain the parallel trends assumption
2. Implement two-way fixed effects regression
3. Conduct event studies
4. Test for pre-trends
5. Apply DiD to policy evaluation

---

## 10.1 The DiD Framework

### The Setup

- Treatment group: California (receives policy in 1988)
- Control group: Other states (no policy)
- Pre-period: Before 1988
- Post-period: After 1988

### The Parallel Trends Assumption

The key assumption is that without treatment, California's smoking trend would have been the same as other states:

$$E[Y_{it}(0) - Y_{it-1}(0) | D_i = 1] = E[Y_{it}(0) - Y_{it-1}(0) | D_i = 0]$$

### The DiD Estimator

$$\hat{\tau}_{DiD} = (Y_{post,treat} - Y_{pre,treat}) - (Y_{post,control} - Y_{pre,control})$$

---

## 10.2 Two-Way Fixed Effects

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import linearmodels

np.random.seed(42)
n_states = 40
n_years = 10

# Generate panel data
states = np.repeat(range(n_states), n_years)
years = np.tile(range(n_years), n_states)
treated = (states < 10).astype(int)  # First 10 states treated
post = (years >= 5).astype(int)  # Treatment starts at year 5

# Fixed effects
state_fe = np.random.normal(0, 1, n_states)[states]
time_fe = np.random.normal(0, 0.5, n_years)[years]

# Outcome with treatment effect
treatment_effect = 2.0
y = 10 + 0.5 * years + state_fe + time_fe + \
    treatment_effect * treated * post + np.random.normal(0, 1, n_states * n_years)

# Create DataFrame
df = pd.DataFrame({
    'state': states,
    'year': years,
    'treated': treated,
    'post': post,
    'y': y
})

# DiD regression
df['did'] = df['treated'] * df['post']
X = sm.add_constant(df[['treated', 'post', 'did']])
model = sm.OLS(df['y'], X).fit()
print("DiD estimate:", round(model.params['did'], 3))
print("True effect:", treatment_effect)

---

## 10.3 Event Study

In [ ]:
# Event study: leads and lags
for t in range(-5, 6):
    df[f'lead_lag_{t}'] = (df['treated'] == 1) & (df['year'] - 5 == t)

# Include leads (omitting the period before treatment as reference)
leads_lags = [f'lead_lag_{t}' for t in range(-5, 6) if t != -1]
X_event = sm.add_constant(df[leads_lags])
model_event = sm.OLS(df['y'], X_event).fit()

# Plot coefficients
import matplotlib.pyplot as plt

coef = [model_event.params[f'lead_lag_{t}'] for t in range(-5, 6) if t != -1]
se = [model_event.bse[f'lead_lag_{t}'] for t in range(-5, 6) if t != -1]

plt.figure(figsize=(10, 6))
plt.errorbar(range(-5, 6), coef, yerr=1.96 * np.array(se), fmt='o-', capsize=5)
plt.axvline(x=0, color='red', linestyle='--', label='Treatment')
plt.axhline(y=0, color='gray', linestyle='-', alpha=0.5)
plt.xlabel('Time relative to treatment')
plt.ylabel('Coefficient')
plt.title('Event Study')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---

## 10.4 Testing Pre-Trends

If pre-treatment coefficients are significantly different from zero, the parallel trends assumption may be violated.

---

## 10.5 Common Mistakes

1. **Parallel trends violation**: Always plot pre-trends
2. **Anticipation effects**: Treatment may affect outcomes before implementation
3. **Spillovers**: Treatment in one unit affects others
4. **Time-varying confounders**: Factors that change differently across groups

---

## 10.6 Knowledge Check

### Multiple Choice

1. **The parallel trends assumption states:**
   - A) Trends are the same before and after treatment
   - B) Treatment and control groups have the same trends without treatment
   - C) There are no trends
   - D) Trends are parallel after treatment

2. **DiD estimates the causal effect when:**
   - A) Parallel trends hold
   - B) Pre-trends are zero
   - C) Both A and B
   - D) Neither A nor B

3. **An event study tests:**
   - A) Post-treatment effects
   - B) Pre-treatment trends
   - C) Both A and B
   - D) Neither A nor B

4. **Anticipation effects cause:**
   - A) Underestimation
   - B) Overestimation
   - C) No bias
   - D) Bias in unknown direction

5. **Spillovers violate:**
   - A) SUTVA
   - B) Parallel trends
   - C) Independence
   - D) Positivity

### Short Answer

6. **Explain why DiD is called a "natural experiment."**

7. **What happens if pre-trends are not parallel?**

8. **How can you test for anticipation effects?**

9. **Describe the role of the control group in DiD.**

10. **Give an example of a policy evaluation using DiD.**

---

## 10.7 Summary

1. **DiD** compares changes over time between treatment and control groups
2. **Parallel trends** is the key identifying assumption
3. **Event studies** test for pre-trends
4. **Two-way fixed effects** is the standard implementation
5. **Spillovers** and **anticipation effects** are threats to validity

---

## 10.8 Further Reading

- Angrist, J.D. & Pischke, J.S. (2009). *Mostly Harmless Econometrics*. Princeton University Press.
- Callaway, B. & Sant'Anna, P.H. (2021). "Difference-in-Differences with Multiple Time Periods." *Journal of Econometrics*.